# **Module 4: Problem Framing and Scoping**  
The provided notebook uses the ProPublica COMPAS Dataset to guide students through the iterative process of framing a data science research question. Rather than treating a research question (RQ) as a static starting point, this module walks students through the process of refining and crafting a generalizable and reproducible RQ through Exploratory Data Analysis (EDA).

**The Purpose of the Notebook:**
To provide students with a hands-on experience in human-centered and responsible data science, where the primary task is interrogating the data, assumptions, and social context to ask better, more responsible questions about algorithmic decision-making and fairness.

This code book is used to help students with:

1. Initial Problem Framing: Students begin by drafting a "Naive RQ" based on their intuition about recidivism prediction, fairness, and the COMPAS dataset.
1. Guided EDA: Using targeted visualizations and summary statistics, students identify where variable relationships challenge assumptions about algorithmic fairness in criminal justice contexts.
1. The Iteration Loop: A guided step by step process to rewrite their RQ based on their EDA findings. Moving from "Is this predictive model fair?" to more robust and nuanced research questions like "Under what conditions is the model unfair and what impact does that have on the societal and systemic level?"
1. Fairness, Generalizability & Reproducibility Audit: Prompts that encourage students to think critically about dataset limitations, historical and systemic bias, missing context, measurement choices, reproducibility, and what next steps in the data science life cycle would look like.
1. Ethics and Human Impact: Students are encouraged to reflect on the real-world implications of predictive systems in criminal justice, including how data-driven tools can reinforce or mitigate inequalities
The code can be adapted as instructors see fit, including lengthening the example, creating more/different features, increasing the complexity of the model, and/or using your own dataset.

In [ ]:
# Import packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm

In [ ]:
# Load dataset
# For privacy purposes this data set is being read in without personal identifiers
# To make pre-processing easier, the data is being read in with only the columns used in the ProPublica COMPAS evaluation
df = pd.read_csv("compas-scores-two-years.csv", usecols=['age', 'sex', 'c_charge_degree', 'race', 'age_cat',
                                                         'score_text', 'priors_count', 'days_b_screening_arrest',
                                                         'decile_score', 'is_recid', 'two_year_recid', 'c_jail_in', 'c_jail_out'])


### Compas Data



The original data set has 53 columns included in this are personal identifiers and other information the propublica team did not use for their analysis. As this notebook seeks to replicate and explore their process this data set has been limited to the 12 variables used in the original COMPAS data analysis.

A data frame with 7214 rows and 12 variables:

For this analysis the following columns are used, their descriptions are included for clarity.
  
* (integer) age: The age of defendants.

* (factor) c_charge_degree : The charge degree of defendants. F: Felony M: Misdemeanor

* (factor) race: The race of defendants.

* (factor) age_cat: The age category of defendants.

* (factor) score_text: The score category of defendants.

* (factor) sex: The sex of defendants.

* (integer) priors_count: The prior criminal records of defendants.

* (integer) days_b_screening_arrest: The count of days between screening date and (original) arrest date. If they are too far apart, that may indicate an error. If the value is negative, that indicate the screening date happened before the arrest date.

* (integer) decile_score: Indicate the risk of recidivism (Min=1, Max=10)

* (factor) two_year_recid: Binary variable indicate whether defendant is rearrested at within two years.

* (numeric) length_of_stay: The count of days stay in jail.

This information about the data set comes from:
Pfisterer, F., Siyi, W., & Lang, M. (n.d.). COMPAS dataset. In mlr3fairness. https://mlr3fairness.mlr-org.com/reference/compas.html

### Pre-Processing

Pre-processing
Identifying columns are removed

Removed the outliers for abs(days_b_screening_arrest) >= 30 and <= -30.

Removed observations where is_recid != -1.

Removed observations where c_charge_degree != "O".

Removed observations where score_text != 'N/A'.

Factorize the features that are categorical.

Add length of stay (c_jail_out - c_jail_in) in the dataset.

From the original study preprocessing information:

However not all of the rows are useable for the first round of analysis.

There are a number of reasons remove rows because of missing data:

If the charge date of a defendants Compas scored crime was not within 30 days from when the person was arrested, we assume that because of data quality reasons, that we do not have the right offense.
We coded the recidivist flag -- is_recid -- to be -1 if we could not find a compas case at all.
In a similar vein, ordinary traffic offenses -- those with a c_charge_degree of 'O' -- will not result in Jail time are removed (only two of them).
We filtered the underlying data from Broward county to include only those rows representing people who had either recidivated in two years, or had at least two years outside of a correctional facility.

Pre-processing Resource: https://github.com/propublica/compas-analysis/blob/master/Compas%20Analysis.ipynb

The simple preprocessing list information comes from:  
Pfisterer, F., Siyi, W., & Lang, M. (n.d.). COMPAS dataset. In mlr3fairness. https://mlr3fairness.mlr-org.com/reference/compas.html

In [ ]:
from pandas.core.arrays import categorical
# Pre-process the data

# Remove outliers
df = df[abs(df['days_b_screening_arrest']) <= 30]
df = df[abs(df['days_b_screening_arrest']) >= -30]

# Remove observations where is_recid == -1
df = df[df['is_recid'] != -1]

# Remove observations where c_charge_degree == "O"
df = df[df['c_charge_degree'] != "O"]

# Remove observations where score_text == "N/A"
df = df[df['score_text'] != "N/A"]

# Add length of stay
# Convert to datetime
df["c_jail_in"] = pd.to_datetime(df["c_jail_in"])
df["c_jail_out"] = pd.to_datetime(df["c_jail_out"])


df['length_of_stay'] = df['c_jail_out'] - df['c_jail_in']

# Check the new dataframe
print("Dataset shape:",  df.shape)

Dataset shape: (6172, 14)


In [ ]:
# Quick overview of the data

# Create readable tables to easily glimpse the dataset
print("--- DATASET OVERVIEW ---")
df.info()

print("\n--- CENTRAL TENDENCY FOR CONTINUOUS VARIABLES ---")
# Keep .describe() wrapped in print()
print(df.describe())

print("\n--- GLIMPSE OF THE DATASET ---")
print(df.head())

--- DATASET OVERVIEW ---
<class 'pandas.core.frame.DataFrame'>
Index: 6172 entries, 0 to 7213
Data columns (total 14 columns):
 #   Column                   Non-Null Count  Dtype          
---  ------                   --------------  -----          
 0   sex                      6172 non-null   object         
 1   age                      6172 non-null   int64          
 2   age_cat                  6172 non-null   object         
 3   race                     6172 non-null   object         
 4   decile_score             6172 non-null   int64          
 5   priors_count             6172 non-null   int64          
 6   days_b_screening_arrest  6172 non-null   float64        
 7   c_jail_in                6172 non-null   datetime64[ns] 
 8   c_jail_out               6172 non-null   datetime64[ns] 
 9   c_charge_degree          6172 non-null   object         
 10  is_recid                 6172 non-null   int64          
 11  score_text               6172 non-null   object         
 12  

## Naive Research Question:

Looking only at the information above about the dataset and your understanding about the criminal justice system, automated processes using existing and historical criminal justice data for training, and good ML and data science practices, craft a research question you think you could answer with this dataset.

**A good research question:**
1. Should ask about something we don’t already know
1. Can be easily understood
1. Is specific enough to be a data science project, not too vague
1. Is broad enough to be meaningful beyond the data, not so specific that the answer is easily found and has little meaning beyond finding the answer

Write your research question here:

## Exploratory Data Analysis (EDA)
Explore the ProPublica COMPAS dataset to better understand the variables, distributions, and potential relationships related to recidivism risk assessment and criminal justice outcomes. Use visualizations and summary statistics to investigate patterns to help you develop a meaningful research question.

### Consider:
1. What does the distribution of COMPAS risk scores look like? Explore using a histogram. Is the distribution skewed or concentrated around certain score ranges?
1. Explore the counts of categorical variables such as race, gender, age category, charge degree, and recidivism status. Use count plots to examine the distributions. How might imbalances in these categories affect interpretations of fairness or bias?
1. Look at correlations between continuous variables such as age, prior offenses, jail stay length, and COMPAS scores? Explore these relationships using a heatmap or pair plot.
1. What do relationships between variables reveal about patterns in criminal justice outcomes, risk assessment, and potential biases in the COMPAS system? How do these findings compare to your assumptions about fairness and recidivism prediction?

This EDA incorporates the exploration done by the original ProPublica team for the COMPAS data which can be found here: @url https://github.com/propublica/compas-analysis/blob/master/Compas%20Analysis.ipynb

Student To Dos:


In [ ]:
# Look at the racial balance in the dataset
# Find the percentage of the dataset each race makes up

In [ ]:
# Look at the distribution of  scores
# Create a box plot of COMPAS scores by race

In [ ]:
# Look at the count and distribution for recidivism score df['score_text'] by race, and by sex

In [ ]:
# Look at the relationship between sex and race
# Use a bar chart

In [ ]:
# Look at the relationship between decile scores and recidivism
# Create a boxplot of decile scores by recidivism rates

In [ ]:
# Look at one multivariate relationship
# Ex. decile score, race, and sex

In [ ]:
# Look at how accruately the COMPAS score aligns with the actual two-year recidivism outcomes?
# Bar chart of recidivism rates by score_text

In [ ]:
# Run logistic regression
# See what relationship there is between input variables and COMPAS score

# Binary target
df['score_factor'] = np.where(df['score_text'] != 'Low', 1, 0)

# Select predictors
X = df[['age_cat', 'race', 'priors_count',
        'sex', 'c_charge_degree', 'two_year_recid']]

# One-hot encode the categorical variables
X = pd.get_dummies(X,drop_first=True)

# Target Variable
y = df['score_factor']

# Convert to numeric
X = X.astype(float)
y = y.astype(float)


# Add intercept
X =sm.add_constant(X)

# Run logistic regression
model = sm.Logit(y, X).fit()

# Results
print(model.summary())



Optimization terminated successfully.
         Current function value: 0.499708
         Iterations 6
                           Logit Regression Results                           
Dep. Variable:           score_factor   No. Observations:                 6172
Model:                          Logit   Df Residuals:                     6160
Method:                           MLE   Df Model:                           11
Date:                Wed, 27 May 2026   Pseudo R-squ.:                  0.2729
Time:                        14:43:57   Log-Likelihood:                -3084.2
converged:                       True   LL-Null:                       -4241.7
Covariance Type:            nonrobust   LLR p-value:                     0.000
                              coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------
const                      -0.8271      0.091     -9.132      0.000      -1.005   

In [ ]:
# Create a confusion matrix crosstab to calculate False Positives and False Negatives
# We will compare African-American and Caucasian defendants

# Filter dataset for the two largest cohorts for this specific analysis
df_filtered = df[df['race'].isin(['African-American', 'Caucasian'])]

# Create a crosstab of actual recidivism vs. predicted high-risk score
confusion = pd.crosstab(df_filtered['race'],
                        [df_filtered['two_year_recid'], df_filtered['score_factor']],
                        rownames=['Race'],
                        colnames=['Actual Recid', 'Predicted High Risk'])

print("--- CONFUSION MATRIX ---")
print(confusion)
print("\n")

# Calculate False Positive Rate (FPR) and False Negative Rate (FNR)
# FPR = FP / (FP + TN) -> Predicted high risk, but did not recidivate
# FNR = FN / (FN + TP) -> Predicted low risk, but did recidivate

for race in ['African-American', 'Caucasian']:
    TN = confusion.loc[race, (0, 0)]
    FP = confusion.loc[race, (0, 1)]
    FN = confusion.loc[race, (1, 0)]
    TP = confusion.loc[race, (1, 1)]

    FPR = FP / (FP + TN)
    FNR = FN / (FN + TP)

    print(f"{race} FPR (Wrongly labeled High Risk): {FPR:.2%}")
    print(f"{race} FNR (Wrongly labeled Low Risk): {FNR:.2%}")

--- CONFUSION MATRIX ---
Actual Recid           0         1      
Predicted High Risk    0    1    0     1
Race                                    
African-American     873  641  473  1188
Caucasian            999  282  408   414


African-American FPR (Wrongly labeled High Risk): 42.34%
African-American FNR (Wrongly labeled Low Risk): 28.48%
Caucasian FPR (Wrongly labeled High Risk): 22.01%
Caucasian FNR (Wrongly labeled Low Risk): 49.64%


## Research Question
Now that you have further explored the data set, take what you have learned about different relationships in the data that you would want to research further. Iterate on your original research question and craft a strong research question for the COMPAS dataset.


Write your findings here:

### Step 1: Identifying Confounding Human and System Factors

Look at your visualizations showing relationships between COMPAS risk scores and variables such as race, age, prior offenses, and recidivism outcomes.

Consider:
1. How do the observed patterns challenge your intuitive assumptions about what should determine a person’s risk score?
For example: do higher scores always align with higher actual recidivism rates?
1. What “human” or institutional factors might explain why two individuals with similar histories receive different COMPAS scores?
1. How might systemic factors in policing, arrests, or sentencing influence variables like prior offenses or charge degree, and in turn affect the COMPAS score?
1. What does the COMPAS score reflect about decision-making in the criminal justice system such as policing patterns, historical bias, data collection limits, instead of just an individuals behavior?
1. How might differences across demographic groups race, age, gender reveal patterns in how risk is assigned rather than actual differences in criminal behavior?
1. How do the observed patterns in the False Positive and False Negative rates challenge your intuitive assumptions about algorithmic neutrality? What “human” or institutional factors might explain why two individuals with similar histories, but different racial backgrounds, bear different burdens of model error?

Reflect on your findings here:

### Step 2: Under What Conditions?
Look at your original RQ, does it ask "Does X cause Y"?  
A robust RQ should ask "Under what conditions does X relate to Y"  
How can you rewrite your RQ to account for confounding variables you have discovered in your EDA?

Consider:  
With the COMPAS dataset it is also important to think about the context in which this data was collected, and the data the system was trained on. The purpose of such a tool is meant to remove human bias from sentencing however from your EDA what might you want to ask about this dataset and this algorithmic tool?

Write your more robust RQ here:

### Step 3: Stakeholders and Impact
Now that you have a robust RQ you need to ask "Why does this matter and who is affected by it?"

Data science does not exist in a vacuum. The outputs of risk assessment tools like COMPAS are used in real criminal justice decisions that directly affect people’s lives.  

Consider the stakeholders in the COMPAS system:
* Defendants, individuals being assessed:
Their COMPAS score can influence bail decisions, sentencing, parole eligibility, and perceived risk—even though the score is only an algorithmic prediction, not a fact.
* Judges, parole boards, and court systems:
They use COMPAS scores as one input among many, but could overweight or under-contextualize the score when making high-stakes decisions.
* Communities and families:
Especially those disproportionately impacted by policing and incarceration practices, where systemic bias already exists.
* Policing and data collection systems:
Historical arrest data shape the inputs used to generate COMPAS scores, which can reinforce existing disparities.
* Algorithmic technology companies and policymakers:
Those who design, validate, or regulate risk assessment tools have incentives around accuracy, efficiency, fairness, and legal compliance.

Look at your original RQ: Who is most affected by the answer to your question about COMPAS scores and recidivism?
Could your findings be used to justify continued use of algorithmic risk scoring without scrutiny?
Could they be used to improve transparency, reduce bias, and support fairer decision-making?

Iterate on your RQ by adding in considerations of human impact.

Write your new RQ that incorporates human impact:

### Consider:

Now that you have explored the ProPublica COMPAS Dataset and gone through the process of refining your research question, let’s think about two final aspects of good research.

**Generalizability:**
Can your research question be extended to other datasets or similar systems? For example, would your findings about risk scores and fairness likely apply to other algorithmic decision-making tools used in criminal justice, healthcare, child werlfare, or lending systems? What parts of your analysis are specific to COMPAS, and what parts might reflect broader patterns in predictive modeling and human decision-making?

**Reproducibility:**
means that your research is not just understandable it is repeatable, verifiable, and transparent enough that another researcher could follow your exact reasoning from data to conclusion.

In this analysis you are both participating in replication, this dataset, pre=-processing, and many of the steps of EDA are replicating the original ProPublica COMPAS analysis. Your RQ however is building on this research to investigate somehting new.

Think about:

How would you conduct the research for your RQ in a way that others could replicate or verify your results? Could another researcher follow your EDA steps and reach the same refined RQ? Could they see how you got to your initial findings?

Write down a few of the steps you would take to make sure your research can be reproduced.